<a href="https://colab.research.google.com/github/ibtihalalf/Sdaia-Bootcamp/blob/main/C4/M1/Ex1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Task 1**

In [2]:
pip install langchain langchain-openai python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 13.7 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1


In [3]:
import os

try:
    # In Colab? read from userdata (secrets)
    from google.colab import userdata
    ON_COLAB = True
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
except ImportError:
    # Load `.env` file (locally)
    from dotenv import load_dotenv
    load_dotenv(override=True)

In [9]:
import os
import json
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

def create_model(temp):
    return ChatOpenAI(
        model="nvidia/nemotron-3-nano-30b-a3b:free",
        temperature=temp,
        base_url="https://openrouter.ai/api/v1",
        api_key=os.getenv("OPENROUTER_API_KEY"),
    )

low_temp = 0
high_temp = 1

low_model = create_model(low_temp)
high_model = create_model(high_temp)

prompt = """
Explain what machine learning is.
Make it short and poetic.

Return ONLY valid JSON in this format:
{
  "answer": "...",
  "style": "poetic"
}
"""

low_response = low_model.invoke(prompt)
high_response = high_model.invoke(prompt)

low_result = {
    "temperature": low_temp,
    "response": json.loads(low_response.content)
}

high_result = {
    "temperature": high_temp,
    "response": json.loads(high_response.content)
}

print("LOW TEMPERATURE:")
print(json.dumps(low_result, indent=2, ensure_ascii=False))

print("\nHIGH TEMPERATURE:")
print(json.dumps(high_result, indent=2, ensure_ascii=False))

LOW TEMPERATURE:
{
  "temperature": 0,
  "response": {
    "answer": "Machine learning is a whispering garden where data seeds sprout patterns, learning to dance with the unknown.",
    "style": "poetic"
  }
}

HIGH TEMPERATURE:
{
  "temperature": 1,
  "response": {
    "answer": "It teaches machines to hear data's whispers and grow silent insight.",
    "style": "poetic"
  }
}


The low-temperature response is more detailed and structured, providing a clearer explanation.

The high-temperature response is shorter and more creative, but less descriptive.

This shows that lower temperature leads to more consistent and predictable outputs,
while higher temperature increases creativity but reduces detail and consistency.

**Task 2**

In [11]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

load_dotenv()

# 1) Define structured output
class Sentiment(BaseModel):
    sentiment: str = Field(description="Must be one of: positive, neutral, negative")

# 2) Create model
model = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

structured_model = model.with_structured_output(Sentiment)

# 3) Test sentences
sentences = [
    # Positive
    "Kindness creates lasting joy.",
    "Success rewards persistent effort.",
    "I love sunlight. It warms the skin.",

    # Negative
    "Pessimistic all the time.",
    "The storm caused damage!",

    # Neutral
    "The clock ticks steadily."
]

# 4) Run
for text in sentences:
    result = structured_model.invoke(text)
    print(f"Text: {text}")
    print(f"Sentiment: {result.sentiment}")
    print("-" * 40)

Text: Kindness creates lasting joy.
Sentiment: positive_reflection
----------------------------------------
Text: Success rewards persistent effort.
Sentiment: positive_motivational_response
----------------------------------------
Text: I love sunlight. It warms the skin.
Sentiment: positive
----------------------------------------
Text: Pessimistic all the time.
Sentiment: empathetic
----------------------------------------
Text: The storm caused damage!
Sentiment: concerned and helpful
----------------------------------------
Text: The clock ticks steadily.
Sentiment: neutral
----------------------------------------


**Task 3**

In [13]:
import os
from typing import List, Literal
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

load_dotenv()

class Categorization(BaseModel):
    tags: List[Literal["cars", "shopping", "sports", "study", "work"]] = Field(
        description="One or more tags from the allowed list only."
    )

model = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

structured_model = model.with_structured_output(Categorization)

sentences = [
    "That restoration looks incredible; you have a real talent for mechanics.",
    "I found the perfect gift today! The staff was incredibly helpful.",
    "Great game today! Your teamwork was the key to that victory.",
    "Learning together helped me finally grasp these concepts. Thank you!"
]

for text in sentences:
    result = structured_model.invoke(
        f"""
        Categorize this sentence using only these tags:
        cars, shopping, sports, study, work.

        Return one or more relevant tags.

        Sentence: {text}
        """
    )

    print(f"Text: {text}")
    print(f"Tags: {result.tags}")
    print("-" * 50)

Text: That restoration looks incredible; you have a real talent for mechanics.
Tags: ['cars']
--------------------------------------------------
Text: I found the perfect gift today! The staff was incredibly helpful.
Tags: ['shopping']
--------------------------------------------------
Text: Great game today! Your teamwork was the key to that victory.
Tags: ['sports']
--------------------------------------------------
Text: Learning together helped me finally grasp these concepts. Thank you!
Tags: ['study']
--------------------------------------------------


The model was restricted to return only predefined tags.
Because the output is a list, the model can return multiple categories when needed.
Structured output helps keep the result controlled and consistent.